# ViSta — worked examples, one situation at a time

ViSta stacks interferometric observations of many faint galaxies **in the uv
plane** (on the visibilities, before any image is made) and measures the
stacked line and continuum.

This notebook is made to **learn how to use it**. Every chapter is one
situation, on observations simulated for it, so that the right answer is
known; every cell does one thing, and the text says what and why.

| | Situation | What it shows |
|---|---|---|
| 1 | One galaxy | what the simulator makes, with and without noise |
| 2 | The basic case | the same galaxy at twelve redshifts: stack, line, continuum, results |
| 3 | The window, the normalisations, the noise | what each option of the measurement changes |
| 4 | Not subtracting the continuum | fitting it with the line, or leaving it |
| 5 | A weak continuum under a strong line | the continuum takes the size of the line |
| 6 | A weak line on a strong continuum | the line takes the size of the continuum |
| 7 | A few galaxies that count much more | natural and democratic weighting; lensed galaxies |
| 8 | Points and extended sources | which source model to use; what a mixed sample does |
| 9 | Two lines far apart | measured each on its own |
| 10 | Two lines that blend | fitted together |
| 11 | A line that is not a Gaussian | the flux in the window |
| 12 | A source that is not a Gaussian | reading the baseline plot |
| 13 | The continuum alone | a stack without any line |

On Colab: *Runtime → Run all* (about 20 minutes on the CPU). Every number is
in the code cells: change it and run again from that cell.

## 0. Setup

The code is cloned from GitHub and its C++ kernel is built (a couple of
minutes, once). For a private repository, add a GitHub token to the Colab
secrets (key icon on the left) as `GITHUB_TOKEN`. Outside Colab, set
`VISTA_REPO` to a built checkout.

In [ ]:
REPO_URL = "https://github.com/martitors/ViSta-HPC.git"
BRANCH = "extract_opt"

import os, sys, glob, subprocess, logging
IN_COLAB = "google.colab" in sys.modules
REPO = os.environ.get("VISTA_REPO")            # a local checkout, to run this outside Colab
if REPO is None:
    REPO = "/content/ViSta-HPC" if IN_COLAB else os.path.abspath("ViSta-HPC")
    if not os.path.isdir(REPO):
        url = REPO_URL
        if IN_COLAB:
            try:                                 # private repository: token from the Colab secrets
                from google.colab import userdata
                url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
            except Exception:
                pass
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
    else:                                        # a clone of an earlier session: update it
        head = lambda: subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                                      capture_output=True, text=True).stdout
        before = head()
        subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", BRANCH])
        if head() != before:                     # new code: build the kernel again, and
            for so in glob.glob(f"{REPO}/vista/ms_ops*.so"):     # forget what was imported
                os.remove(so)
            for name in [m for m in sys.modules if m.split(".")[0] in ("vista", "simulate")]:
                del sys.modules[name]
            if "ms_ops" in sys.modules:
                print("The code was updated: after this cell, Runtime > Restart session, "
                      "then Run all")

if IN_COLAB:
    subprocess.run(f"pip install -q -r {REPO}/requirements.txt pybind11", shell=True, check=True)
if not glob.glob(f"{REPO}/vista/ms_ops*.so"):          # build the kernel once
    subprocess.run(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release > /dev/null"
                   f" && cmake --build {REPO}/build -j 2 > /dev/null"
                   f" && cmake --install {REPO}/build > /dev/null", shell=True, check=True)
sys.path[:0] = [REPO, os.path.join(REPO, "examples")]

import numpy as np
import matplotlib.pyplot as plt
import simulate                                        # examples/simulate.py: the simulator
import vista
from vista import ViSta

logging.getLogger("daskms").setLevel(logging.ERROR)    # quiet about empty optional columns
WORK = "/content/vista_demo" if IN_COLAB else os.path.abspath("vista_demo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("ViSta", vista.__version__, "from", REPO, "· working in", WORK)

A one-line helper for this notebook, to print a flux with its error:

In [ ]:
def flux(result, key="flux"):
    """'0.400 ± 0.050 Jy km/s' from result[key] (small values in mJy)."""
    f = result[key]
    k, unit = (1e3, "m" + f["unit"]) if abs(f["value"]) < 0.01 else (1, f["unit"])   # Jy -> mJy
    return f"{f['value'] * k:.3f} ± {f['error'] * k:.3f} {unit}"

## 1. One galaxy

`examples/simulate.py` writes Measurement Sets like those of a real
observation, so that ViSta reads them as it would read yours. For every
galaxy it makes:

* **the telescope**: an ALMA-like array of 43 antennas within `rmax` metres
  of its centre, observing for 20 minutes; the longer a baseline, the finer
  the details it sees;
* **the line**: a Gaussian in velocity, with integrated flux `line_flux`
  [Jy km/s] and width `fwhm_kms`, at the frequency where the redshift `z`
  puts it;
* **the continuum**: `cont_flux` [Jy], the dust emission under the line;
* **the shape**: a round Gaussian `size_arcsec` across (FWHM), or a point
  (`size_arcsec=0`). An extended source is fainter on the long baselines;
* **the position**: `offset_arcsec` from the centre of the field, where the
  antennas are less sensitive (the *primary beam*);
* **the noise**: `noise` [Jy] on every visibility, correlated between
  adjacent channels as in real data. The weights written in the file
  underestimate it by a factor `kappa`, as real weights often do.

Here is one galaxy, written twice: without noise and with it.

In [ ]:
LINE = 345.79599                       # rest frequency of CO(3-2), GHz

galaxy = dict(z=2.5, rest_freq_ghz=LINE,
              line_flux=0.4, fwhm_kms=300,       # the line: 0.4 Jy km/s, 300 km/s wide
              cont_flux=2e-4,                    # the continuum: 0.2 mJy
              size_arcsec=0.8,                   # the shape
              offset_arcsec=(6, 3),              # 6" east and 3" north of the field centre
              noise=0.1, rmax=600)               # the noise and the size of the array

clean = simulate.make_ms("one_clean.ms", noiseless=True, **galaxy)     # without noise
noisy = simulate.make_ms("one_noisy.ms", **galaxy)                     # as observed

**Top**: the image of the channel at the centre of the line (the circle is
the resolution). **Bottom**: the spectrum at the position of the galaxy, the
real part (Re) of the visibilities averaged over all the baselines.

The two green curves are the true signal. The **solid** one is what these
baselines should see: without noise the black spectrum lies exactly on it.
The **dashed** one is the total flux of the galaxy, higher: the source is
extended (0.8″) and the long baselines see only part of it. The total flux
is what a baseline of zero length would measure, and what ViSta estimates
by fitting how the flux falls with the baseline length.

With the noise of a real observation neither the image nor the spectrum
show the galaxy: this is the galaxy that stacking is for.

In [ ]:
simulate.plot_galaxy(clean, noisy)

## 2. The basic case: the same galaxy at twelve redshifts

### 2.1 The sample

`simulate.make_sample` writes one observation per galaxy and the *input
list* that ViSta reads. Here **the twelve galaxies are the same galaxy**,
put at twelve redshifts between 2 and 3:

* the same line (0.4 Jy km/s at the reference redshift, 300 km/s wide) and
  the same continuum (0.1 mJy);
* the same size: all are **extended**, 0.8″ across;
* every observation has its band centred a little differently
  (`band_offset_kms`), so the edges of the stacked spectrum are covered by
  fewer galaxies;
* each sits somewhere within 20″ of the centre of its field.

In [ ]:
truth = simulate.make_sample(
    "sample", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=300,                               # the same line for all
    cont_flux=1e-4,                                            # the same continuum, Jy
    size_arcsec=0.8,                                           # the same size, 0.8" across
    band_offset_kms=lambda rng: rng.uniform(-350, 350),        # bands centred differently
    offset_arcsec=lambda rng: tuple(rng.uniform(-15, 15, 2)),  # position in the field
    noise=0.07, kappa=1.3, rmax=600, seed=9)

print(simulate.describe(truth))

The input list has one line per dataset: the Measurement Set, the redshift
and the position of the galaxy. It is all ViSta needs to know.

In [ ]:
print(open(truth["input_list"]).read())

**The same galaxy looks different at different redshifts.** The farther it
is, the fainter its line (left). Before combining them, ViSta brings every
galaxy to the flux it would have at one reference redshift, `z_ref` (the
median of the sample unless you give it). After this correction the twelve
lines are **one and the same** (right): the stack is then the average of
twelve measurements of the same thing.

In [ ]:
simulate.plot_redshift_correction(truth)

What every observation sees on its own: the spectrum at the position of the
galaxy (black) and its true line (green). In none of them can the line be
measured.

In [ ]:
simulate.plot_sample(truth)

**The expected answer.** Because the sample is simulated, we know what a
perfect measurement of the stack would give. With real data there is no
such thing: it is here only to check the measurements.

In [ ]:
expected = simulate.expected(truth)
print(f"expected line flux of the stack: {expected['flux'][0]:.3f} Jy km/s")
print(f"expected continuum:              {expected['continuum'] * 1e3:.3f} mJy")

### 2.2 The stack

Two calls describe it:

* `ViSta(...)`: the input list and the name of what is written;
* `v.stack(...)`: the rest frequency of the line (`line_freq`, GHz, or a
  string as `"345.796GHz"`), the window that holds the line
  (`line_window=500`: from −500 to +500 km/s) and the band to keep
  (`velocity_range=900`: ±900 km/s).

`v.run()` then, for every dataset: brings the frequencies to the rest frame
of the galaxy, moves the phase centre onto the galaxy, puts the channels on
a common grid, and fits and subtracts the continuum on the channels outside
the line window. The result is one dataset, `stack.zarr`.

In [ ]:
v = ViSta(truth["input_list"], output_name="stack")
v.stack(line_freq=LINE, line_window=500, velocity_range=900)
v.run()

**What stacking does, in an image.** One galaxy of the stack, then all
twelve: the line appears at the centre, where every galaxy was moved.
(These images are a quick look; the measurements are made on the
visibilities.)

In [ ]:
one = vista.dirty_image("stack.zarr", line_freq=LINE, velocity=(-150, 150), sources=[5],
                        fov_arcsec=16)
vista.plot_image(one, title="One galaxy");

In [ ]:
all12 = vista.dirty_image("stack.zarr", line_freq=LINE, velocity=(-150, 150), fov_arcsec=16)
vista.plot_image(all12, title="The stack of twelve");

### 2.3 Measuring the line

`v.extract()` asks for the measurement, here with all its defaults;
`v.run(only="measure")` makes it on the stack that is already there.
Printing the result says what was measured **and why each choice was made**.

In [ ]:
v.extract()
r = v.run(only="measure")
print(r)

How to read it:

* **Line**: the S/N of the line, and how sure the detection is once the
  search over centre and width is accounted for.
* **flux**: the number to quote. Two fluxes are always measured: the **area
  of the Gaussian fit** and the **area under the spectrum in the window**.
  The Gaussian one is quoted unless the two disagree (chapter 11).
* **source**: the model of the sky. A Gaussian source is used only if it
  fits clearly better than a point source (Δχ² ≥ 9).
* **Continuum**: what was subtracted, measured on its own.
* **Sizes**: the size of the source measured on the line and on the
  continuum, and whether they agree.
* **Reference**: the weighting and the redshift the fluxes refer to.

**The spectrum of the line.** The real part (Re) of the stacked spectrum
(black), its noise (grey band), the fit (red) and the window (orange). On
the left the two fluxes, the quoted one in bold; then centre, width and
peak of the Gaussian, and the S/N. Below, the residuals in units of the
noise, and how many galaxies contribute at every velocity: fewer at the
edges, where the noise grows.

In [ ]:
r.plot("line")

**The line against the baseline length.** The line flux measured in rings
(*annuli*) of baseline length. A point source would give the same flux on
all of them; here it falls on the long baselines: the stacked source is
extended. Red: the Gaussian source; green dashed: the point source; in bold
the one chosen. The flux is where the model meets zero baseline. The grey
crosses are the imaginary part (Im), which holds only noise.

In [ ]:
r.plot("line_uv")

**All the numbers.** The result is read like a dictionary, `r["flux"]`:
every quantity with its value, error and unit (`r.keys()` lists them).

In [ ]:
r["flux"]

In [ ]:
print("flux from the fit:   ", flux(r, "flux_fit"))
print("flux in the window:  ", flux(r, "flux_window"))
print("centre, width:       ", f"{r['centre']['value']:+.0f}, {r['fwhm']['value']:.0f} km/s")
print("S/N, significance:   ", f"{r['snr']:.1f}, {r['significance']:.1f} sigma")
print("size of the source:  ", r["size"])
print("source model:        ", r["model"])
print("noise per channel:   ", r["noise"])

### 2.4 The continuum

With the default `continuum="subtract"` the continuum is taken out of every
galaxy before stacking, so the spectrum above holds the line only, and the
continuum is **measured on its own**.

**The spectrum of the continuum**: the channels outside the line window
(black), with the level measured (blue).

In [ ]:
r.plot("continuum")

**The continuum against the baseline length**, as for the line. This
continuum is weak: on its own it cannot tell whether the source is extended
(the points are too noisy). But the line says it is, and these are the same
galaxies: the continuum is fitted **with the size of the line** (red, in
bold). Chapter 5 looks at this choice.

In [ ]:
r.plot("continuum_uv")

In [ ]:
r["continuum"]

The two sizes, each measured on its own, and how far apart they are in
units of their errors:

In [ ]:
r["sizes"]

## 3. The window, the normalisations, the noise

Each cell changes **one option** of the measurement. `v.extract(...)`
replaces all the options, and `run(only="measure")` repeats the measurement
in seconds, without stacking again.

### 3.1 Quoting the flux in the window

`flux_measure="area"` quotes the sum of the spectrum in the window instead
of the Gaussian fit (now in bold on the plot). It assumes nothing about the
shape of the line, and has a larger error: it also sums the channels that
hold only noise.

In [ ]:
v.extract(flux_measure="area")
area = v.run(only="measure", tag="area")       # tag: its files do not replace the others
area.plot("line")

### 3.2 A narrower window

The window of ±500 km/s is much wider than the line (300 km/s FWHM, all of
it within ±300). `window=320` sums only where the line is: the flux in the
window keeps its value and its error gets smaller. The fit does not change.

In [ ]:
v.extract(window=320)
w320 = v.run(only="measure", tag="w320")
w320.plot("line")

A window **too narrow** cuts the line: the flux in the window drops, the
one of the fit does not move.

In [ ]:
v.extract(window=150)
w150 = v.run(only="measure", tag="w150")
w150.plot("line")

In [ ]:
print("window ±500:", flux(r, "flux_window"))
print("window ±320:", flux(w320, "flux_window"))
print("window ±150:", flux(w150, "flux_window"))
print("expected:   ", f"{expected['flux'][0]:.3f} Jy km/s")

### 3.3 Without the redshift correction

`z_norm=False` stacks the fluxes as observed, each at its own distance. The
number no longer refers to any redshift, and it depends on which redshifts
happen to be in the sample.

In [ ]:
v.extract(z_norm=False)
no_z = v.run(only="measure", tag="no_z")
print("with the correction (default):", flux(r))
print("without:                      ", flux(no_z))
print("expected at z_ref:            ", f"{expected['flux'][0]:.3f} Jy km/s")

### 3.4 Without the primary-beam correction

An antenna is less sensitive away from the centre of its field: these
galaxies, up to 20″ away, are seen at 70–100 % of their flux.
`pb_corr=False` leaves that in, and the stack is low.

In [ ]:
v.extract(pb_corr=False)
no_pb = v.run(only="measure", tag="no_pb")
print("with the correction (default):", flux(r))
print("without:                      ", flux(no_pb))
print("expected:                     ", f"{expected['flux'][0]:.3f} Jy km/s")

### 3.5 The noise

The noise is not taken from the weights of the files: it is measured from
the scatter of the visibilities. `r.plot("noise")` shows the checks:

* left: the noise per channel from the scatter and from the weights. Here
  the weights are 30 % too optimistic (we simulated them so);
* centre: the stacked values away from the line, divided by their noise:
  they follow the bell curve if the noise is right;
* right: the same search for a line repeated on pure noise (grey), and the
  S/N measured (red line).

In [ ]:
r.plot("noise")

## 4. Not subtracting the continuum

`continuum` in `v.stack` says what to do with the continuum:

* `"subtract"` (default): taken out of every galaxy before stacking, and
  measured on its own. Line and continuum can have different sizes. The
  choice for most cases.
* `"fit"`: left in, and fitted **together with the line** as a constant
  level. For when few channels are free of the line (a broad line, a narrow
  band). It uses one size for both.
* `"none"`: nothing is done. Only when the continuum is negligible.

The stack keeps the data before the subtraction, so no new stack is needed:
only the measurement is repeated.

### 4.1 Fitted with the line

The spectrum now shows the line **on top of** the continuum (blue dashed).

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900, continuum="fit")
v.extract()
fitted = v.run(only="measure", tag="fitted")
fitted.plot("line")

### 4.2 Left in the data

Here there is a continuum, and nothing accounts for it: the fit takes it
for part of the line and the flux is too high. ViSta notices: it measures
the level outside the window (blue dashed) and the report warns.

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900, continuum="none")
left = v.run(only="measure", tag="left")
left.plot("line")

In [ ]:
print("subtracted (default): ", flux(r))
print("fitted with the line: ", flux(fitted))
print("left in the data:     ", flux(left))
print("expected:             ", f"{expected['flux'][0]:.3f} Jy km/s")

In [ ]:
v.stack(line_freq=LINE, line_window=500, velocity_range=900)     # back to the subtraction

## 5. A weak continuum under a strong line

This is the sample of chapter 2: the line is well detected and shows that
the source is extended; the continuum is detected, but too weakly to
measure a size.

`continuum_model` says which source model the continuum gets, and
`measure="continuum"` repeats **only** the continuum.

### 5.1 The continuum left to itself

`continuum_model="gauss"` fits its own Gaussian. The points are too noisy to
tell a Gaussian from a point (the two curves fit equally well): the size
that comes out is not a measurement, and the flux depends on it.

In [ ]:
v.extract(continuum_model="gauss", measure="continuum")
own = v.run(only="measure", tag="own")
own.plot("continuum_uv")

### 5.2 As a point

`continuum_model="point"`: a smaller error, but the flux is too low,
because the source is not a point.

In [ ]:
v.extract(continuum_model="point", measure="continuum")
as_point = v.run(only="measure", tag="as_point")
as_point.plot("continuum_uv")

### 5.3 With the size of the line

The default (`continuum_model="auto"`) does this by itself when the
continuum cannot measure its size and the line can; `continuum_model="line"`
asks for it. It assumes that gas and dust have the same extent: the report
says so, and gives the two sizes side by side. The three choices give three
different fluxes:

In [ ]:
print("its own Gaussian:        ", flux(own, "continuum"))
print("a point:                 ", flux(as_point, "continuum"))
print("the size of the line:    ", flux(r, "continuum"))
print("expected:                ", f"{expected['continuum'] * 1e3:.3f} mJy")

## 6. A weak line on a strong continuum

The opposite case, on a new sample: a bright continuum (1 mJy) and a faint
line (0.2 Jy km/s), all galaxies 0.8″ across.

In [ ]:
truth_weak = simulate.make_sample(
    "weak", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.2, fwhm_kms=300,           # a faint line
    cont_flux=1e-3,                        # a bright continuum: 1 mJy
    size_arcsec=0.8, noise=0.1, kappa=1.3, rmax=600, seed=22)
expected_weak = simulate.expected(truth_weak)

k = ViSta(truth_weak["input_list"], output_name="weak")
k.stack(line_freq=LINE, line_window=500, velocity_range=900)
k.extract()
weak = k.run()
print(weak)

The continuum measures its size well. The line alone cannot: a Gaussian
source does not fit clearly better than a point. Since the continuum says
that these galaxies are extended, the line is fitted **with the size of the
continuum** (the report says so; `model="continuum"` asks for it).

In [ ]:
weak.plot("continuum_uv")

In [ ]:
weak.plot("line_uv")

In [ ]:
weak.plot("line")

What the line would give as a point source (`model="point"`;
`measure="line"` repeats only the line): too low.

In [ ]:
k.extract(model="point", measure="line")
weak_point = k.run(only="measure", tag="point")
print("with the size of the continuum:", flux(weak))
print("as a point:                    ", flux(weak_point))
print("expected:                      ", f"{expected_weak['flux'][0]:.3f} Jy km/s")

## 7. A few galaxies that count much more

`weighting_scheme="natural"` (the default) weights every visibility by its
noise: the deepest observations count most, and the noise of the stack is
the lowest possible. That is right when all the galaxies are alike. It is
not when a few of them are different **and** observed much deeper.

We add to the twelve galaxies of chapter 2 **two lensed ones**: the same
galaxy magnified 5 times, observed 4 times deeper.

In [ ]:
lensed = simulate.make_sample(
    "lensed", n_sources=2, z_range=(2.3, 2.6), rest_freq_ghz=LINE,
    line_flux=0.4 * 5, fwhm_kms=300, cont_flux=1e-4 * 5,       # magnified 5 times
    size_arcsec=0.8, z_ref=truth["z_ref"],
    noise=0.07 / 4, kappa=1.3, rmax=600, seed=7)        # 4 times deeper

# one list with the fourteen, and the magnification of each in the NORM column
all14 = simulate.write_list("all14.txt", truth["sources"] + lensed["sources"],
                            z_ref=truth["z_ref"], norm=[1] * 12 + [5] * 2)
print(open("all14.txt").read())

### 7.1 The two lensed galaxies alone

(`z_ref` is given, so that every stack of this chapter refers to the same
redshift.)

In [ ]:
l = ViSta(lensed["input_list"], output_name="lensed_stack")
l.stack(line_freq=LINE, line_window=500, velocity_range=900)
l.extract(z_ref=truth["z_ref"])
two = l.run()
two.plot("line")

### 7.2 The fourteen, natural weighting

The stack looks like the two lensed galaxies: they hold most of the weight.

In [ ]:
a = ViSta(all14["input_list"], output_name="all14")
a.stack(line_freq=LINE, line_window=500, velocity_range=900)
a.extract(z_ref=truth["z_ref"])
natural = a.run()
natural.plot("line")

### 7.3 The fourteen, democratic weighting

`weighting_scheme="democratic"` gives every galaxy the same weight: the
stack is the plain mean of the fourteen, with more noise.

In [ ]:
a.extract(z_ref=truth["z_ref"], weighting_scheme="democratic")
democratic = a.run(only="measure", tag="democratic")
democratic.plot("line")

The three spectra together: the natural stack lies on the lensed pair, the
democratic one far below. Neither is wrong: they answer different
questions, and each has its own expected value.

In [ ]:
vista.compare({"two lensed alone": two, "fourteen, natural": natural,
               "fourteen, democratic": democratic}, table=False)

In [ ]:
print("two lensed alone:     ", flux(two))
print("fourteen, natural:    ", flux(natural),
      f"  expected {simulate.expected(all14)['flux'][0]:.3f}")
print("fourteen, democratic: ", flux(democratic),
      f"  expected {simulate.expected(all14, weighting='democratic')['flux'][0]:.3f}")

### 7.4 When the magnification is known

`flux_norm=True` divides every galaxy by its `NORM` (here the
magnification). All fourteen are again the same galaxy, and the natural
weighting can use the deep data at full value: the flux of chapter 2, with
a smaller error.

In [ ]:
a.extract(z_ref=truth["z_ref"], flux_norm=True, flux_ref=1)
delensed = a.run(only="measure", tag="delensed")
delensed.plot("line")

In [ ]:
print("twelve unlensed (chapter 2):", flux(r))
print("fourteen, de-lensed:        ", flux(delensed))
print("expected:                   ", f"{expected['flux'][0]:.3f} Jy km/s")

## 8. Points and extended sources

Six galaxies seen as **points** and six **extended** (0.8″), the same line
in all of them. First each kind on its own, to see which source model is
the right one; then the twelve together.

In [ ]:
same = dict(z_range=(2.0, 3.0), rest_freq_ghz=LINE, line_flux=0.4, fwhm_kms=300,
            z_ref=truth["z_ref"], noise=0.05, kappa=1.3, rmax=600)
points = simulate.make_sample("mix_points", n_sources=6, size_arcsec=0.0, seed=31, **same)
extended = simulate.make_sample("mix_extended", n_sources=6, size_arcsec=0.8, seed=32, **same)

### 8.1 The points

The flux is the same on every baseline. The default (`model="auto"`) picks
the point source.

In [ ]:
p = ViSta(points["input_list"], output_name="points")
p.stack(line_freq=LINE, line_window=500, velocity_range=900)
p.extract(z_ref=truth["z_ref"])
pts = p.run()
pts.plot("line_uv")

Forcing a Gaussian (`model="gauss"`) adds a parameter the data do not need:
the same flux within the errors, with a **larger error**.

In [ ]:
p.extract(z_ref=truth["z_ref"], model="gauss", measure="line")
pts_gauss = p.run(only="measure", tag="gauss")
print("point (chosen):  ", flux(pts))
print("Gaussian forced: ", flux(pts_gauss))
print("expected:        ", f"{simulate.expected(points)['flux'][0]:.3f} Jy km/s")

### 8.2 The extended ones

The flux falls on the long baselines, and the default picks the Gaussian.

In [ ]:
e = ViSta(extended["input_list"], output_name="extended")
e.stack(line_freq=LINE, line_window=500, velocity_range=900)
e.extract(z_ref=truth["z_ref"])
ext = e.run()
ext.plot("line_uv")

Forcing a point (`model="point"`) misses the flux that the long baselines
do not see: **too low**, and with an error that looks small.

In [ ]:
e.extract(z_ref=truth["z_ref"], model="point", measure="line")
ext_point = e.run(only="measure", tag="point")
print("Gaussian (chosen): ", flux(ext))
print("point forced:      ", flux(ext_point))
print("expected:          ", f"{simulate.expected(extended)['flux'][0]:.3f} Jy km/s")

### 8.3 The twelve together

A stack of points and extended sources is **neither**: on the baselines it
is a flat level (the points) plus something that falls (the extended ones).
A single Gaussian cannot follow both. It settles on a small size, and the
flux it finds at zero baseline is **too low**.

Look at the short baselines: the points lie above the model. That is the
sign of a sample whose sources have very different sizes.

In [ ]:
mixed = simulate.write_list("mixed.txt", points["sources"] + extended["sources"],
                            z_ref=truth["z_ref"])
m = ViSta(mixed["input_list"], output_name="mixed")
m.stack(line_freq=LINE, line_window=500, velocity_range=900)
m.extract(z_ref=truth["z_ref"])
mix = m.run()
mix.plot("line_uv")

On 60 simulated stacks like this one the flux came out 16 % low on average,
with an error that held the truth only once in seven
([Validation](https://vista-hpc.readthedocs.io/en/latest/extraction/validation.html)).
A single stack can land closer or farther by chance.

What to do with real data: if you can tell the compact galaxies from the
extended ones (from the continuum, from another line, from their images),
stack them separately, as above. If you cannot, the flux of the stack is a
lower limit of this kind, and the plot against the baseline length is where
to see it.

In [ ]:
print("points alone:    ", flux(pts))
print("extended alone:  ", flux(ext))
print("all together:    ", flux(mix))
print("expected (each): ", f"{simulate.expected(mixed)['flux'][0]:.3f} Jy km/s")

## 9. Two lines far apart

A new sample: every galaxy has a second line, half as bright, 1500 km/s from
the first. `line_freq` takes a list, and every line has its window. The
windows do not touch, so **each line is measured on its own**: its own
centre, width, flux and size.

In [ ]:
FAR = LINE * (1 - 1500 / 299792.458)           # 1500 km/s from the first
truth_far = simulate.make_sample(
    "far", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=2,
    nchan=170, band_offset_kms=750,                                # a band that holds both
    extra_lines=[dict(rest_freq_ghz=FAR, flux_ratio=0.5, fwhm_kms=250)])

f = ViSta(truth_far["input_list"], output_name="far")
f.stack(line_freq=[LINE, FAR], line_window=300, velocity_range=(-700, 2200))
f.extract()
far = f.run()
print(far)

In [ ]:
far.plot("line")

In [ ]:
for k, line in enumerate(far["lines"]):
    print(f"line {k + 1}: {line['flux']['value']:.3f} ± {line['flux']['error']:.3f} Jy km/s"
          f"   expected {simulate.expected(truth_far)['flux'][k]:.3f}")

## 10. Two lines that blend

The same, with the second line only 400 km/s away: the two overlap. Their
windows overlap too, so ViSta fits them **together**, one Gaussian each
with the same source size, and separates them.

In [ ]:
NEAR = LINE * (1 - 400 / 299792.458)           # 400 km/s from the first
truth_blend = simulate.make_sample(
    "blend", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=2,
    nchan=150, extra_lines=[dict(rest_freq_ghz=NEAR, flux_ratio=0.5, fwhm_kms=250)])

b = ViSta(truth_blend["input_list"], output_name="blend")
b.stack(line_freq=[LINE, NEAR], line_window=300, velocity_range=(-900, 1300))
b.extract()
blend = b.run()
print(blend)

In [ ]:
blend.plot("line")

In [ ]:
for k, line in enumerate(blend["lines"]):
    print(f"line {k + 1}: {line['flux']['value']:.3f} ± {line['flux']['error']:.3f} Jy km/s"
          f"   expected {simulate.expected(truth_blend)['flux'][k]:.3f}")

## 11. A line that is not a Gaussian

Every galaxy of this sample has a line with broad wings (a third of the
flux in a component 800 km/s wide, as an outflow would give).

The Gaussian fit follows the core and misses the wings: its area is too
small. The sum in the window assumes no shape and catches them. ViSta sees
that the two disagree by more than their errors allow, and quotes **the
flux in the window** (in bold): the report says so, and why.

In [ ]:
truth_wings = simulate.make_sample(
    "wings", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.4, fwhm_kms=250, size_arcsec=0.5, noise=0.05, rmax=600, seed=1,
    nchan=150,                                                             # a wider band
    extra_lines=[dict(rest_freq_ghz=LINE, flux_ratio=0.5, fwhm_kms=800)])  # the wings

w = ViSta(truth_wings["input_list"], output_name="wings")
w.stack(line_freq=LINE, line_window=1000, velocity_range=1400)
w.extract()
wings = w.run()
print(wings)

In [ ]:
wings.plot("line")

In [ ]:
print("flux in the window (quoted):", flux(wings, "flux_window"))
print("flux of the Gaussian fit:   ", flux(wings, "flux_fit"))
print("expected:                   ", f"{simulate.expected(truth_wings)['flux'][0]:.3f} Jy km/s")

## 12. A source that is not a Gaussian

Every galaxy of this sample is a **ring** 3″ across. ViSta models the
stacked source as a point or a round Gaussian: what happens when it is
neither?

In [ ]:
truth_ring = simulate.make_sample(
    "ring", n_sources=12, z_range=(2.0, 3.0), rest_freq_ghz=LINE,
    line_flux=0.6, fwhm_kms=300, size_arcsec=3.0, shape="ring",     # a ring 3" across
    noise=0.04, rmax=600, seed=5)

g = ViSta(truth_ring["input_list"], output_name="ring")
g.stack(line_freq=LINE, line_window=500, velocity_range=900)
g.extract(window=320)
ring = g.run()
ring.plot("image")

The image of the stack shows the ring. Against the baseline length a ring
has a signature: its flux goes **negative** on some baselines, which no
Gaussian can do. The fitted Gaussian follows the short baselines, which set
the total flux, and misses the rest. The flux is still about right, because
it is decided at the short baselines; the size is that of "a Gaussian as
wide as the ring", not a property of the galaxies.

For real data: look at this plot. Points that go systematically below the
model, or below zero, say that the source is not what the model assumes.

In [ ]:
ring.plot("line_uv")

In [ ]:
print("measured:", flux(ring))
print("expected:", f"{simulate.expected(truth_ring)['flux'][0]:.3f} Jy km/s")

## 13. The continuum alone

Sometimes there is no line at all: a continuum survey, galaxies observed in
the same band whatever their redshift. Ten galaxies between z = 1 and 3,
all observed at 100 GHz, with a continuum of 0.3 mJy (at `z_ref`) and no
line.

In [ ]:
truth_cont = simulate.make_sample(
    "cont", n_sources=10, z_range=(1.0, 3.0),
    observed_freq_ghz=100.0,               # the same band for all
    line_flux=0.0, cont_flux=3e-4,         # no line; continuum 0.3 mJy
    size_arcsec=0.6, noise=0.05, rmax=600, nchan=64, seed=13)

`v.stack()` **without `line_freq`** makes the stack of the continuum: every
channel of every dataset is kept, and the bands need not overlap in the
rest frame. `v.extract()` then measures the continuum: its flux density and
the size of the source, with the same rules as before.

In [ ]:
c = ViSta(truth_cont["input_list"], output_name="cont")
c.stack()                                  # no line: the continuum
c.extract()
cont = c.run()
print(cont)

In [ ]:
cont.plot("continuum_uv")

In [ ]:
print("measured:", flux(cont, "continuum"))
print("size:    ", cont["continuum"]["size"])
print("expected:", f"{simulate.expected(truth_cont)['continuum'] * 1e3:.3f} mJy")

## 14. The command line

`python -m vista` (or `run_vista.sh`) has the same options as `ViSta()`,
`stack()` and `extract()`, one flag each; `--extract` asks for the
measurement. It writes the same files: the stack, `*_results.json` with
every number, and the figures as PNG.

In [ ]:
!PYTHONPATH={REPO} python -m vista --input sample/input_list.txt --output-name cli/stack \
    --line-freq 345.796GHz --line-window 500 --velocity-range 900 \
    --extract --window 320

## 15. On your own data

1. Write the input list, one line per dataset:
   `<ms_path> <redshift> <RA> <Dec> [FIELD_ID] [SPW_IDS] [NORM]`, with the
   position of the galaxy (hh:mm:ss.s ±dd:mm:ss.s).
2. `python examples/inspect_ms.py input_list.txt --line-freq 345.796GHz --all`
   reads the bands of the datasets and suggests `velocity_range`,
   `line_window` and `annuli`.
3. Stack and measure as in chapter 2, with a window wide enough for the
   whole line.
4. Read the report (`print(r)`) and look at the plots (`r.plot()` draws the
   line and the continuum, spectrum and baselines; `r.plot("noise")`,
   `r.plot("image")`): residuals around zero, nothing left of the
   continuum, the model on the points, the two sizes in agreement.
5. Change one option at a time, with `run(only="measure")`, as in chapters
   3 to 8. `v.help()` lists every option with its value and what it can be.

The documentation, <https://vista-hpc.readthedocs.io>, explains every option.